# ДЗ9 · Пост-хок прогон на Kaggle T4
В приватный Notebook добавьте приватный Dataset с исходным bundle_posthoc.zip. В настройках выберите GPU T4 x2 и включите Internet для скачивания закреплённых зависимостей и весов. Используется одна T4. Запускайте ячейки сверху вниз. Скачайте ZIP с ответами из Output. Сырые ответы и бандл содержат пользовательские входы. Сохраняйте notebook и Dataset приватными.

In [1]:
# 1. Locate inputs and set output paths.
import os
import sys
from pathlib import Path

DRY_RUN = False
if DRY_RUN:
    workspace = Path(os.environ["HW9_POSTHOC_DRY_WORKSPACE"])
    inputs = workspace / "input"
    bundle = workspace / "bundle"
    results = workspace / "dry_results"
else:
    workspace = Path("/kaggle/working")
    inputs = Path("/kaggle/input")
    bundle = Path("/tmp/hw9_posthoc_bundle")
    results = workspace / "posthoc_results"
results.mkdir(parents=True, exist_ok=True)

In [2]:
# 2. Verify the ZIP or the exact extracted inputs.
import hashlib
import json
from shutil import copyfile
from zipfile import ZipFile

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

expected_archive = "fcd96954ae85ae2b53bec2bff92e9b5750751a3425e1a2c760c1cfeea6a25ee2"
expected_manifest = "ea1aad899d0cb9356dc08a24c40699e7162d62a624171426695caa1a90ebc706"
archives = sorted(inputs.rglob("bundle_posthoc*.zip"))
archive = next((path for path in archives if file_hash(path) == expected_archive), None)
if archive is not None:
    with ZipFile(archive) as zipped:
        manifest_raw = zipped.read("manifest.json")
        assert hashlib.sha256(manifest_raw).hexdigest() == expected_manifest
        manifest = json.loads(manifest_raw)
        members = manifest["file_sha256"]
        assert sorted(zipped.namelist()) == sorted([*members, "manifest.json"])
        assert all(Path(name).name == name and name not in {".", ".."} for name in members)
        for name, checksum in members.items():
            assert hashlib.sha256(zipped.read(name)).hexdigest() == checksum, name
        zipped.extractall(bundle)
    print("Verified input: original ZIP")
else:
    manifests = sorted(inputs.rglob("manifest.json"))
    source_manifest = next((path for path in manifests if file_hash(path) == expected_manifest), None)
    if source_manifest is None:
        raise RuntimeError("No matching HW9 input in /kaggle/input. Check the attached private Dataset and wait until processing finishes.")
    manifest = json.loads(source_manifest.read_bytes())
    members = manifest["file_sha256"]
    assert all(Path(name).name == name and name not in {".", ".."} for name in members)
    for name, checksum in members.items():
        candidate = source_manifest.parent / name
        assert candidate.is_file() and file_hash(candidate) == checksum, f"Missing or changed input: {name}"
    bundle.mkdir(parents=True, exist_ok=True)
    for name in (*members, "manifest.json"):
        copyfile(source_manifest.parent / name, bundle / name)
    print("Verified input: extracted files match the original ZIP manifest")

sys.path.insert(0, str(bundle))
from stage3_runtime import append, check_identity, read_jsonl, run_set
sets = ("main_g", "ablation_g_seed0", "ablation_g_seed1", "ablation_g_seed2")
records = {name: read_jsonl(bundle / f"{name}.jsonl") for name in sets}
assert manifest["prompt_counts"] == {name: 100 for name in sets}
for name, rows in records.items():
    assert [r["id"] for r in rows] == [f"{name}:U{i:03d}" for i in range(1, 101)]
    for row in rows:
        assert hashlib.sha256(row["prompt"].encode()).hexdigest() == row["prompt_sha256"] == manifest["prompt_sha256"][row["id"]]
gen = manifest["generation"]
print("Verified:", sum(map(len, records.values())), "frozen prompts")

Verified input: extracted files match the original ZIP manifest
Verified: 400 frozen prompts


In [3]:
# 3. Match the HW8/HW9 dependencies and verify the pinned checkpoint.
if not DRY_RUN:
    import subprocess
    import time
    hardware = subprocess.run(["nvidia-smi"], check=True, capture_output=True, text=True).stdout
    print(hardware)
    (results / "nvidia-smi.log").write_text(hardware, encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6",
                    "huggingface_hub==0.36.0", "hf_xet==1.2.0", "psutil==7.2.2"], check=True)
    import torch
    import transformers
    from huggingface_hub import snapshot_download

    assert torch.cuda.is_available()
    gpu = torch.cuda.get_device_name()
    assert "T4" in gpu, f"Select Colab T4, current GPU: {gpu}"
    started = time.perf_counter()
    weights = Path(snapshot_download(repo_id=gen["model"], revision=gen["revision"],
                                     allow_patterns=list(manifest["model_files_sha256"]), token=False))
    assert weights.name == gen["revision"]
    weight_hashes = {name: file_hash(weights / name) for name in manifest["model_files_sha256"]}
    assert weight_hashes == manifest["model_files_sha256"]
    identity = {"bundle_sha256": expected_archive, "revision": gen["revision"], "gpu": gpu,
                "torch": torch.__version__, "transformers": transformers.__version__,
                "generation": gen, "weight_sha256": weight_hashes}
    check_identity(results / "run_identity.json", identity)
    append(results / "timing.jsonl", {"stage": "checkpoint_and_verification",
                                     "seconds": time.perf_counter() - started})
    print({"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
           "revision": gen["revision"]})

Fri Nov 27 08:30:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.0 which is incompatible.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

... (4 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

{'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'transformers': '4.57.6', 'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'}


In [4]:
# 4. Generate raw responses in frozen order and resume from per-set journals.
if DRY_RUN:
    class Fake:
        calls = 0
        def generate(self, prompt):
            self.calls += 1
            return {"text": "synthetic response", "input_tokens": len(prompt.split()),
                    "output_tokens": 2, "seconds": 0.0}

    trial = records["main_g"][:2]
    path = results / "main_g.jsonl"
    backend = Fake()
    first = run_set(trial[:1], path, backend)
    resumed = run_set(trial, path, backend)
    reopened = run_set(trial, path, backend)
    assert [(x["skipped"], x["new"]) for x in (first, resumed, reopened)] == [(0, 1), (1, 1), (2, 0)]
    assert backend.calls == 2 and [r["id"] for r in read_jsonl(path)] == [r["id"] for r in trial]
    print("Dry-run:", backend.calls, "synthetic calls; resumed:", resumed, "reopened:", reopened)
else:
    from gpu_llm import GPUQwen
    import traceback
    lengths = {name: len(read_jsonl(results / f"{name}.jsonl")) for name in sets}
    try:
        backend = None
        if any(lengths[name] < len(records[name]) for name in sets):
            started = time.perf_counter()
            backend = GPUQwen(weights, gen)
            append(results / "timing.jsonl", {"stage": "model_load", "dtype": backend.metadata["dtype"],
                                             "seconds": time.perf_counter() - started})
        for name in sets:
            journal = results / f"{name}.jsonl"
            completed = lengths[name]
            append(results / "timing.jsonl", {"stage": name, "event": "start", "completed": completed})
            try:
                summary = run_set(records[name], journal, backend)
            finally:
                saved = read_jsonl(journal)
                append(results / "timing.jsonl", {"stage": name, "event": "end",
                                                 "completed": len(saved), "new": len(saved) - completed,
                                                 "new_generation_seconds": sum(row["seconds"] for row in saved[completed:])})
            print(name, summary)
    except Exception:
        with (results / "errors.log").open("a", encoding="utf-8") as log:
            traceback.print_exc(file=log)
        raise

I0000 00:00:1795768361.318462      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

main_g {'skipped': 0, 'new': 100, 'generation_seconds': 397.33756505400015}
ablation_g_seed0 {'skipped': 0, 'new': 100, 'generation_seconds': 400.6480023680002}
ablation_g_seed1 {'skipped': 0, 'new': 100, 'generation_seconds': 400.0342227040003}
ablation_g_seed2 {'skipped': 0, 'new': 100, 'generation_seconds': 399.8822906099988}


In [5]:
# 5. Package the journals for download from Kaggle Output.
if not DRY_RUN:
    from zipfile import ZipFile, ZIP_DEFLATED
    download = workspace / "hw9_posthoc_colab_results.zip"
    with ZipFile(download, "w", ZIP_DEFLATED) as zipped:
        for source in sorted(results.iterdir()):
            if source.is_file() and source.suffix in {".json", ".jsonl", ".log"}:
                zipped.write(source, source.name)
    print("Download from Output:", download.name)

Download from Output: hw9_posthoc_colab_results.zip
